In [0]:
CREATE OR REPLACE TEMP TABLE tb_reservas_prueba AS
SELECT *
FROM workspace.hotel_db.tb_reservas
WHERE 1 = 0;

INSERT INTO tb_reservas_prueba
SELECT *
FROM workspace.hotel_db.tb_reservas
WHERE hotel = 'City Hotel'
  AND is_canceled = 1;

num_affected_rows,num_inserted_rows
33102,33102


In [0]:
UPDATE tb_reservas_prueba
SET babies = 0;

num_affected_rows
33102


In [0]:
SELECT
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    CONCAT(
        LPAD(CAST(arrival_date_day_of_month AS STRING), 2, '0'),
        '-', CAST(arrival_date_month AS STRING),
        '-', CAST(arrival_date_year AS STRING)
    ) AS arrival_date_string
FROM workspace.hotel_db.tb_reservas;

arrival_date_year,arrival_date_month,arrival_date_day_of_month,arrival_date_string
2015,July,1,01-July-2015
2015,July,1,01-July-2015
2015,July,1,01-July-2015
2015,July,1,01-July-2015
2015,July,1,01-July-2015
2015,July,1,01-July-2015
2015,July,1,01-July-2015
2015,July,1,01-July-2015
2015,July,1,01-July-2015
2015,July,1,01-July-2015


In [0]:
SELECT *
FROM workspace.hotel_db.tb_reservas
WHERE distribution_channel LIKE 'T%';

hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
Resort Hotel,0,14,2015,July,27,1,0,2,2,0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,240,NULL,0,Transient,98.0,0,1,Check-Out,2015-07-03
Resort Hotel,0,14,2015,July,27,1,0,2,2,0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,240,NULL,0,Transient,98.0,0,1,Check-Out,2015-07-03
Resort Hotel,1,85,2015,July,27,1,0,3,2,0,0,BB,PRT,Online TA,TA/TO,0,0,0,A,A,0,240,NULL,0,Transient,82.0,0,1,Canceled,2015-05-06
Resort Hotel,1,75,2015,July,27,1,0,3,2,0,0,HB,PRT,Offline TA/TO,TA/TO,0,0,0,D,D,0,15,NULL,0,Transient,105.5,0,0,Canceled,2015-04-22
Resort Hotel,1,23,2015,July,27,1,0,4,2,0,0,BB,PRT,Online TA,TA/TO,0,0,0,E,E,0,240,NULL,0,Transient,123.0,0,0,Canceled,2015-06-23
Resort Hotel,0,35,2015,July,27,1,0,4,2,0,0,HB,PRT,Online TA,TA/TO,0,0,0,D,D,0,240,NULL,0,Transient,145.0,0,0,Check-Out,2015-07-05
Resort Hotel,0,68,2015,July,27,1,0,4,2,0,0,BB,USA,Online TA,TA/TO,0,0,0,D,E,0,240,NULL,0,Transient,97.0,0,3,Check-Out,2015-07-05
Resort Hotel,0,18,2015,July,27,1,0,4,2,1,0,HB,ESP,Online TA,TA/TO,0,0,0,G,G,1,241,NULL,0,Transient,154.77,0,1,Check-Out,2015-07-05
Resort Hotel,0,37,2015,July,27,1,0,4,2,0,0,BB,PRT,Online TA,TA/TO,0,0,0,E,E,0,241,NULL,0,Transient,94.71,0,0,Check-Out,2015-07-05
Resort Hotel,0,68,2015,July,27,1,0,4,2,0,0,BB,IRL,Online TA,TA/TO,0,0,0,D,E,0,240,NULL,0,Transient,97.0,0,3,Check-Out,2015-07-05


In [0]:
SELECT
    COUNT(*) AS total_reservas,
    SUM(total_of_special_requests) AS total_solicitudes_especiales,
    AVG(adr) AS adr_promedio
FROM workspace.hotel_db.tb_reservas;

total_reservas,total_solicitudes_especiales,adr_promedio
119390,68215,101.83112153446665


In [0]:
CREATE OR REPLACE TEMP TABLE tb_reservas_prueba_filtrada AS
SELECT *
FROM tb_reservas_prueba
WHERE adr IS NULL OR adr <> 0.0;
DROP TEMP TABLE tb_reservas_prueba;
CREATE TEMP TABLE tb_reservas_prueba AS
SELECT * FROM tb_reservas_prueba_filtrada;
DROP TEMP TABLE tb_reservas_prueba_filtrada;

In [0]:
SELECT distribution_channel, COUNT(*) AS cantidad_reservas
FROM workspace.hotel_db.tb_reservas
GROUP BY distribution_channel
ORDER BY cantidad_reservas DESC;

distribution_channel,cantidad_reservas
TA/TO,97870
Direct,14645
Corporate,6677
GDS,193
Undefined,5


In [0]:
SELECT
    adr,
    CASE
        WHEN adr IS NULL THEN 'Sin dato'
        WHEN adr < 50 THEN 'Económico'
        WHEN adr BETWEEN 50 AND 150 THEN 'Estándar'
        ELSE 'Premium'
    END AS categoria_precio
FROM workspace.hotel_db.tb_reservas;

adr,categoria_precio
0.0,Económico
0.0,Económico
75.0,Estándar
75.0,Estándar
98.0,Estándar
98.0,Estándar
107.0,Estándar
103.0,Estándar
82.0,Estándar
105.5,Estándar


In [0]:
SELECT
  arrival_date_year,
  arrival_date_month,
  arrival_date_day_of_month,
  lead_time,
  DATE_SUB(
    TO_DATE(
      CONCAT(arrival_date_year, '-', arrival_date_month, '-', arrival_date_day_of_month),
      'yyyy-MMMM-d'
    ),
    TRY_CAST(lead_time AS INT)
  ) AS booking_date
FROM workspace.hotel_db.tb_reservas;

arrival_date_year,arrival_date_month,arrival_date_day_of_month,lead_time,booking_date
2015,July,1,342,2014-07-24
2015,July,1,737,2013-06-24
2015,July,1,7,2015-06-24
2015,July,1,13,2015-06-18
2015,July,1,14,2015-06-17
2015,July,1,14,2015-06-17
2015,July,1,0,2015-07-01
2015,July,1,9,2015-06-22
2015,July,1,85,2015-04-07
2015,July,1,75,2015-04-17


In [0]:
SELECT lead_time, CEIL(lead_time / 30.0) AS meses_anticipacion_techo
FROM workspace.hotel_db.tb_reservas;

lead_time,meses_anticipacion_techo
342,12
737,25
7,1
13,1
14,1
14,1
0,0
9,1
85,3
75,3
